In [1]:
#add libraries for tab
import pandas as pd #tab
import geopandas as gpd #vector
#mount the drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# import the geojson file '/content/drive/MyDrive/ESIIL2025MSU/Project Data/norm_irrigatedacres_gdf.geojson'
irrigatedacres_gdf = gpd.read_file('/content/drive/MyDrive/ESIIL2025MSU/Project Data/norm_irrigatedacres_gdf.geojson')

In [3]:
# create a new geodataframe where rows where 'COUNTY' = DOUGLAS or 'COUNTY' = EAGLE or 'COUNTY' = LOGAN
ia_casestudy = irrigatedacres_gdf[(irrigatedacres_gdf['COUNTY'] == 'DOUGLAS') | (irrigatedacres_gdf['COUNTY'] == 'EAGLE') | (irrigatedacres_gdf['COUNTY'] == 'LOGAN')].copy()

# create a new dataframe with the columns: 'YEAR', 'LOGAN', 'DOUGLAS', 'EAGLE'

# Populate the dataframe by iterating through the unique years in ia_casestudy
new_rows = []
for year in ia_casestudy['Year'].unique():
    year_data = ia_casestudy[ia_casestudy['Year'] == year]
    new_row = {'YEAR': year}
    for county in ['DOUGLAS', 'EAGLE', 'LOGAN']:
        county_data = year_data[year_data['COUNTY'] == county]
        if not county_data.empty:
            new_row[county] = county_data['norm_irrigatedacres'].iloc[0]
        else:
            new_row[county] = None # or pd.NA, depending on how you want to handle missing data
    new_rows.append(new_row)

nia_DouglasEagleLogan = pd.DataFrame(new_rows)
display(ia_casestudy.head())
display(nia_DouglasEagleLogan)
#save the dataframe to a csv file
nia_DouglasEagleLogan.to_csv('/content/drive/MyDrive/ESIIL2025MSU/Project Data/nia_DouglasEagleLogan.csv', index=False)

,COUNTY,Year,all_irrigatedfarms_acres,all_irrigatedland_acres,area_acres,norm_allacres,norm_irrigatedacres,irrigated_ratio,geometry
45,EAGLE,1982,197846.0,29751.0,1.082704e+06,18.273316,2.747841,15.037453,"POLYGON ((-107.11359 39.49009, -107.11359 39.4..."
46,EAGLE,1987,201559.0,25763.0,1.082704e+06,18.616253,2.379504,12.781865,"POLYGON ((-107.11359 39.49009, -107.11359 39.4..."
47,EAGLE,1992,192624.0,24093.0,1.082704e+06,17.791005,2.225261,12.507787,"POLYGON ((-107.11359 39.49009, -107.11359 39.4..."
48,EAGLE,1997,136903.0,16637.0,1.082704e+06,12.644540,1.536615,12.152400,"POLYGON ((-107.11359 39.49009, -107.11359 39.4..."
49,EAGLE,2002,82567.0,9306.0,1.082704e+06,7.625996,0.859514,11.270847,"POLYGON ((-107.11359 39.49009, -107.11359 39.4..."


,YEAR,DOUGLAS,EAGLE,LOGAN
0,1982,1.039718,2.747841,7.957884
1,1987,0.857841,2.379504,7.998085
2,1992,0.620345,2.225261,8.854065
3,1997,0.675779,1.536615,9.241769
4,2002,0.655015,0.859514,9.357801
5,2007,0.640183,1.027797,8.486841
6,2012,0.312583,1.176960,7.955684
7,2017,0.423637,1.365100,9.060654
8,2022,0.146651,1.330926,7.714649


In [4]:
# Repeat the ANOVA analysis for the three time periods using statsmodels.formula.api ols and statsmodels.stats.anova anova_lm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import numpy as np

# Load the CSV file
file_path = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/nia_DouglasEagleLogan.csv'
df = pd.read_csv(file_path)

# Reshape the data for ANOVA (long format)
df_melted = df.melt(id_vars=['YEAR'], var_name='COUNTY', value_name='Normalized_Acres')

# ANOVA for years 1985-2001
df_1985_2001 = df_melted[(df_melted['YEAR'] >= 1985) & (df_melted['YEAR'] <= 2001)].copy()
# Drop rows with missing values for this period
df_1985_2001.dropna(inplace=True)

if not df_1985_2001.empty:
    print("ANOVA for years 1985-2001:")
    # Perform one-way ANOVA
    model_1985_2001 = ols('Normalized_Acres ~ C(COUNTY)', data=df_1985_2001).fit()
    anova_table_1985_2001 = anova_lm(model_1985_2001, typ=2)
    print(anova_table_1985_2001)
else:
    print("No data available for ANOVA for years 1985-2001 after dropping NaNs.")

# ANOVA for years 2002-2022
df_2002_2022 = df_melted[(df_melted['YEAR'] >= 2002) & (df_melted['YEAR'] <= 2022)].copy()
# Drop rows with missing values for this period
df_2002_2022.dropna(inplace=True)

if not df_2002_2022.empty:
    print("\nANOVA for years 2002-2022:")
    # Perform one-way ANOVA
    model_2002_2022 = ols('Normalized_Acres ~ C(COUNTY)', data=df_2002_2022).fit()
    anova_table_2002_2022 = anova_lm(model_2002_2022, typ=2)
    print(anova_table_2002_2022)
else:
    print("No data available for ANOVA for years 2002-2022 after dropping NaNs.")

# ANOVA for all years 1985-2022
df_1985_2022 = df_melted[(df_melted['YEAR'] >= 1985) & (df_melted['YEAR'] <= 2022)].copy()
# Drop rows with missing values for this period
df_1985_2022.dropna(inplace=True)

if not df_1985_2022.empty:
    print("\nANOVA for years 1985-2022:")
    # Perform one-way ANOVA
    model_1985_2022 = ols('Normalized_Acres ~ C(COUNTY)', data=df_1985_2022).fit()
    anova_table_1985_2022 = anova_lm(model_1985_2022, typ=2)
    print(anova_table_1985_2022)
else:
    print("No data available for ANOVA for years 1985-2022 after dropping NaNs.")

ANOVA for years 1985-2001:
               sum_sq   df           F    PR(>F)
C(COUNTY)  109.680509  2.0  264.582536  0.000001
Residual     1.243625  6.0         NaN       NaN

ANOVA for years 2002-2022:
               sum_sq    df           F        PR(>F)
C(COUNTY)  200.010928   2.0  515.042925  2.331667e-12
Residual     2.330030  12.0         NaN           NaN

ANOVA for years 1985-2022:
               sum_sq    df           F        PR(>F)
C(COUNTY)  309.133590   2.0  613.824889  2.347880e-19
Residual     5.287995  21.0         NaN           NaN


In [5]:
# Use the To_csv() method to save each ANOVA table DataFrame to a CSV file with an appropriate filename

# Define the directory to save the CSV files
output_dir = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/'

# Save the ANOVA tables to CSV files
if 'anova_table_1985_2001' in locals():
    anova_table_1985_2001.to_csv(output_dir + 'nia_anova_85_01.csv')

if 'anova_table_2002_2022' in locals():
    anova_table_2002_2022.to_csv(output_dir + 'nia_anova_02_22.csv')

if 'anova_table_1985_2022' in locals():
    anova_table_1985_2022.to_csv(output_dir + 'nia_anova_85_22.csv')

print("ANOVA tables saved to CSV files in", output_dir)

ANOVA tables saved to CSV files in /content/drive/MyDrive/ESIIL2025MSU/Project Data/


In [10]:
# Select the relevant columns and drop rows with missing values for the specified columns
douglas_data = irrigatedacres_gdf[irrigatedacres_gdf['COUNTY'] == 'DOUGLAS'].copy()
plot_data = douglas_data[['Year', 'all_irrigatedfarms_acres', 'all_irrigatedland_acres']].dropna().copy()

# Convert acres to thousands of acres
plot_data['all_irrigatedfarms_acres_thousands'] = plot_data['all_irrigatedfarms_acres'] / 1000
plot_data['all_irrigatedland_acres_thousands'] = plot_data['all_irrigatedland_acres'] / 1000

# Sort by year for proper plotting
plot_data = plot_data.sort_values(by='Year')

# Create the interactive plot with two y-axes using Plotly Express
# Plotly Express automatically handles creating multiple traces for different y-axes if the data is in long format.
# Since our data is in wide format, we will use Plotly Graph Objects to have more control over the layout.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces for the primary y-axis (all_irrigatedfarms_acres)
fig.add_trace(
    go.Scatter(x=plot_data['Year'], y=plot_data['all_irrigatedfarms_acres_thousands'],
               mode='lines+markers', name='Irrigated Farms Acres', marker=dict(color='teal'), line=dict(color='teal')),
    secondary_y=False,
)

# Add traces for the secondary y-axis (all_irrigatedland_acres)
fig.add_trace(
    go.Scatter(x=plot_data['Year'], y=plot_data['all_irrigatedland_acres_thousands'],
               mode='lines+markers', name='Irrigated Land Acres', marker=dict(color='orange'), line=dict(color='orange')),
    secondary_y=True,
)

# Add figure title
fig.update_layout(
    title_text="Land in Irrigated Agriculture: Douglas County (1982-2022)",
    title_font_size=30, # Increase title font size
    hovermode='x unified', # Improve hover behavior
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=18) # Increase legend font size
    )
)

# Set x-axis title
fig.update_xaxes(title_text="Year", showgrid=True, gridcolor='lightgrey', title_font_size=18, tickfont=dict(size=16))

# Set y-axes titles
fig.update_yaxes(title_text="Acres in Farms with Irrigation Systems (Thousands)", secondary_y=False, showgrid=True, gridcolor='lightgrey', title_font=dict(color='teal', size=18), tickfont=dict(size=16))
fig.update_yaxes(title_text="Irrigated Acres (Thousands)", secondary_y=True, showgrid=True, gridcolor='lightgrey', title_font=dict(color='orange', size=18), tickfont=dict(size=16))


# Set plot background to transparent
fig.update_layout(plot_bgcolor='rgba(0,0,0,0)')


# Show the plot in the notebook
fig.show()

# Define the path to save the HTML file
html_file_path_two_axis = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/douglas_irrigated_acres_two_axis_interactive.html'

# Save the interactive plot as an HTML file
fig.write_html(html_file_path_two_axis)

print(f"Interactive two-axis plot saved as HTML to: {html_file_path_two_axis}")

Interactive two-axis plot saved as HTML to: /content/drive/MyDrive/ESIIL2025MSU/Project Data/douglas_irrigated_acres_two_axis_interactive.html


In [7]:
# Filter the data for EAGLE county
eagle_data = irrigatedacres_gdf[irrigatedacres_gdf['COUNTY'] == 'EAGLE'].copy()

# Select the relevant columns and drop rows with missing values for the specified columns
plot_data_eagle = eagle_data[['Year', 'all_irrigatedfarms_acres', 'all_irrigatedland_acres']].dropna().copy()

# Convert acres to thousands of acres
plot_data_eagle['all_irrigatedfarms_acres_thousands'] = plot_data_eagle['all_irrigatedfarms_acres'] / 1000
plot_data_eagle['all_irrigatedland_acres_thousands'] = plot_data_eagle['all_irrigatedland_acres'] / 1000

# Sort by year for proper plotting
plot_data_eagle = plot_data_eagle.sort_values(by='Year')

# Create figure with secondary y-axis
fig_eagle = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces for the primary y-axis (all_irrigatedfarms_acres)
fig_eagle.add_trace(
    go.Scatter(x=plot_data_eagle['Year'], y=plot_data_eagle['all_irrigatedfarms_acres_thousands'],
               mode='lines+markers', name='Irrigated Farms Acres', marker=dict(color='teal'), line=dict(color='teal')),
    secondary_y=False,
)

# Add traces for the secondary y-axis (all_irrigatedland_acres)
fig_eagle.add_trace(
    go.Scatter(x=plot_data_eagle['Year'], y=plot_data_eagle['all_irrigatedland_acres_thousands'],
               mode='lines+markers', name='Irrigated Land Acres', marker=dict(color='orange'), line=dict(color='orange')),
    secondary_y=True,
)

# Add figure title
fig_eagle.update_layout(
    title_text="Land in Irrigated Agriculture: Eagle County (1982-2022)",
    title_font_size=30, # Increase title font size
    hovermode='x unified', # Improve hover behavior
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=18) # Increase legend font size
    )
)

# Set x-axis title
fig_eagle.update_xaxes(title_text="Year", showgrid=True, gridcolor='lightgrey', title_font_size=18, tickfont=dict(size=16))

# Set y-axes titles
fig_eagle.update_yaxes(title_text="Acres in Farms with Irrigation Systems (Thousands)", secondary_y=False, showgrid=True, gridcolor='lightgrey', title_font=dict(color='teal', size=18), tickfont=dict(size=16))
fig_eagle.update_yaxes(title_text="Irrigated Acres (Thousands)", secondary_y=True, showgrid=True, gridcolor='lightgrey', title_font=dict(color='orange', size=18), tickfont=dict(size=16))

# Set plot background to transparent
fig_eagle.update_layout(plot_bgcolor='rgba(0,0,0,0)')

# Show the plot in the notebook
fig_eagle.show()

# Define the path to save the HTML file
html_file_path_eagle_two_axis = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/eagle_irrigated_acres_two_axis_interactive.html'

# Save the interactive plot as an HTML file
fig_eagle.write_html(html_file_path_eagle_two_axis)

print(f"Interactive two-axis plot for Eagle County saved as HTML to: {html_file_path_eagle_two_axis}")

Interactive two-axis plot for Eagle County saved as HTML to: /content/drive/MyDrive/ESIIL2025MSU/Project Data/eagle_irrigated_acres_two_axis_interactive.html


In [8]:
# prompt: recreate the plotly graph again, using logan county

# Filter the data for LOGAN county
logan_data = irrigatedacres_gdf[irrigatedacres_gdf['COUNTY'] == 'LOGAN'].copy()

# Select the relevant columns and drop rows with missing values for the specified columns
plot_data_logan = logan_data[['Year', 'all_irrigatedfarms_acres', 'all_irrigatedland_acres']].dropna().copy()

# Convert acres to thousands of acres
plot_data_logan['all_irrigatedfarms_acres_thousands'] = plot_data_logan['all_irrigatedfarms_acres'] / 1000
plot_data_logan['all_irrigatedland_acres_thousands'] = plot_data_logan['all_irrigatedland_acres'] / 1000

# Sort by year for proper plotting
plot_data_logan = plot_data_logan.sort_values(by='Year')

# Create figure with secondary y-axis
fig_logan = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces for the primary y-axis (all_irrigatedfarms_acres)
fig_logan.add_trace(
    go.Scatter(x=plot_data_logan['Year'], y=plot_data_logan['all_irrigatedfarms_acres_thousands'],
               mode='lines+markers', name='Irrigated Farms Acres', marker=dict(color='teal'), line=dict(color='teal')),
    secondary_y=False,
)

# Add traces for the secondary y-axis (all_irrigatedland_acres)
fig_logan.add_trace(
    go.Scatter(x=plot_data_logan['Year'], y=plot_data_logan['all_irrigatedland_acres_thousands'],
               mode='lines+markers', name='Irrigated Land Acres', marker=dict(color='orange'), line=dict(color='orange')),
    secondary_y=True,
)

# Add figure title
fig_logan.update_layout(
    title_text="Land in Irrigated Agriculture: Logan County (1982-2022)",
    title_font_size=30, # Increase title font size
    hovermode='x unified', # Improve hover behavior
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=18) # Increase legend font size
    )
)

# Set x-axis title
fig_logan.update_xaxes(title_text="Year", showgrid=True, gridcolor='lightgrey', title_font_size=18, tickfont=dict(size=16))

# Set y-axes titles
fig_logan.update_yaxes(title_text="Acres in Farms with Irrigation Systems (Thousands)", secondary_y=False, showgrid=True, gridcolor='lightgrey', title_font=dict(color='teal', size=18), tickfont=dict(size=16))
fig_logan.update_yaxes(title_text="Irrigated Acres (Thousands)", secondary_y=True, showgrid=True, gridcolor='lightgrey', title_font=dict(color='orange', size=18), tickfont=dict(size=16))

# Set plot background to transparent
fig_logan.update_layout(plot_bgcolor='rgba(0,0,0,0)')

# Show the plot in the notebook
fig_logan.show()

# Define the path to save the HTML file
html_file_path_logan_two_axis = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/logan_irrigated_acres_two_axis_interactive.html'

# Save the interactive plot as an HTML file
fig_logan.write_html(html_file_path_logan_two_axis)


print(f"Interactive two-axis plot for Logan County saved as HTML to: {html_file_path_logan_two_axis}")

Interactive two-axis plot for Logan County saved as HTML to: /content/drive/MyDrive/ESIIL2025MSU/Project Data/logan_irrigated_acres_two_axis_interactive.html


In [12]:
import plotly.express as px

# Create the plot
fig = px.line(ia_casestudy, x='Year', y='all_irrigatedland_acres', color='COUNTY',
              title='Irrigated Agriculture by County (1982-2022)',
              labels={'all_irrigatedland_acres': 'Irrigated Land Acres', 'Year': 'Year'},
              markers=True,
              category_orders={"COUNTY": ["DOUGLAS", "EAGLE", "LOGAN"]})

# Update layout for a cleaner look
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Irrigated Land Acres",
    legend_title="County",
    title_font_size=30, # Increase title font size
    plot_bgcolor='rgba(0,0,0,0)',
    hovermode='x unified', # Improve hover behavior
    legend_title_font_size=20, # Increase legend title font size
    legend_font_size=18, # Increase legend font size
    xaxis=dict(showgrid=True, gridcolor='lightgrey', title_font_size=20, tickfont=dict(size=18)), # Increase x-axis label font size
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title_font_size=20, tickfont=dict(size=18)), # Increase y-axis label and tick font size
    legend=dict(font=dict(size=18))
)

# Show the plot
fig.show()

# Define the path to save the HTML file
html_file_path_all_counties_land = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/all_counties_irrigated_land_acres.html'

# Save the interactive plot as an HTML file
fig.write_html(html_file_path_all_counties_land)

print(f"Interactive plot for all counties saved as HTML to: {html_file_path_all_counties_land}")

Interactive plot for all counties saved as HTML to: /content/drive/MyDrive/ESIIL2025MSU/Project Data/all_counties_irrigated_land_acres.html
